# Alpaca Dataset Generator with Llama 3.2

This notebook provides a complete workflow to generate a custom instruction-following dataset in the Alpaca format. It uses the `meta-llama/Llama-3.2-1B` model to intelligently create instruction-output pairs from your own source documents (`.txt` or `.pdf`).

**Key Features:**
- **LLM-Powered Generation:** Leverages a powerful language model to create relevant and coherent data.
- **Kaggle Optimized:** Designed to run on Kaggle notebooks, using GPU acceleration and secrets for Hugging Face authentication.
- **Multi-Format Input:** Accepts both `.txt` and `.pdf` files from a Kaggle dataset as source material.
- **User-Friendly:** Prompts for the dataset path and the desired number of examples.
- **Multiple Export Options:** Saves the final dataset in JSON (Alpaca format), TXT, and PDF.

## 1. Setup and Installations

First, we install all the required Python libraries. `transformers`, `datasets`, `accelerate`, and `torch` are standard for working with Hugging Face models. `bitsandbytes` is used for model quantization (to save memory), `pypdf` is for reading PDF files, and `fpdf2` is for saving the output as a PDF.

In [ ]:
!pip install transformers datasets accelerate bitsandbytes pypdf torch fpdf2

## 2. Imports and Hugging Face Login

Next, we import the necessary modules and set up our Hugging Face authentication. To download the Llama-3.2 model, you need a Hugging Face token.

In [ ]:
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
import getpass
from pypdf import PdfReader
from fpdf import FPDF

### Hugging Face Authentication
This cell attempts to log in to Hugging Face using a token stored in Kaggle Secrets. If you're not running this on Kaggle or haven't set up the secret, it will prompt you to enter your token manually.

**To add your token on Kaggle:**
1. Go to the "Add-ons" menu and select "Secrets".
2. Create a new secret with the label `HUGGINGFACE_TOKEN` and paste your token as the value.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
    login(token=hf_token)
    print("Hugging Face token found and login successful!")
except ImportError:
    print("Kaggle secrets not found. Please enter your Hugging Face token manually.")
    hf_token = getpass.getpass("Enter your Hugging Face Token: ")
    login(token=hf_token)

## 3. Load Data from Kaggle Dataset

These functions handle the reading of source documents. You can place `.txt` and `.pdf` files in a Kaggle dataset and this code will extract the text from them.

In [ ]:
def read_text_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading text file {file_path}: {e}")
        return ""

def read_pdf_file(file_path):
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text
        return text
    except Exception as e:
        print(f"Error reading PDF file {file_path}: {e}")
        return ""

Run the cell below and provide the path to your Kaggle dataset when prompted. For example: `/kaggle/input/my-text-files`.

In [ ]:
dataset_path = input("Enter the path to your Kaggle dataset (e.g., /kaggle/input/your-dataset-name): ")

full_text = ""
if not os.path.isdir(dataset_path):
    print(f"Error: The path '{dataset_path}' does not exist or is not a directory.")
else:
    print(f"Reading files from: {dataset_path}")
    for filename in os.listdir(dataset_path):
        file_path = os.path.join(dataset_path, filename)
        if filename.endswith('.txt'):
            print(f"- Reading text file: {filename}")
            full_text += read_text_file(file_path) + "\n\n"
        elif filename.endswith('.pdf'):
            print(f"- Reading PDF file: {filename}")
            full_text += read_pdf_file(file_path) + "\n\n"

print("\nFinished reading all files.")
print(f"Total characters read: {len(full_text)}")

## 4. Load the Language Model (Llama-3.2-1B)

Here, we load the `meta-llama/Llama-3.2-1B` model. We use 4-bit quantization (`BitsAndBytesConfig`) to significantly reduce the model's memory footprint, which is crucial for running on Kaggle's free-tier GPUs.

In [ ]:
model_id = "meta-llama/Llama-3.2-1B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model... This may take a few minutes.")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto", # Automatically select the device (GPU if available)
)

print("Model and tokenizer loaded successfully!")

## 5. Generate Alpaca-Formatted Dataset

This is the core of the notebook. The `generate_instruction_output` function takes a piece of text (context), feeds it to the LLM with a specific prompt, and parses the model's response to extract an instruction and an output.

In [ ]:
def generate_instruction_output(context, model, tokenizer):
    prompt_template = f"""You are an expert in creating high-quality instruction-following datasets. 
Based on the following context, generate a single, concise instruction and a corresponding detailed output in the style of the Alpaca dataset. 
The instruction should be a task that can be completed using the information in the context. 
The output should be the answer to that instruction.

**Context:**
\"\"\"
{context}
\"\"\"

**Generated Instruction:**
[Your generated instruction here]

**Generated Output:**
[Your generated output here]"""

    # Format the prompt with the provided context
    prompt = prompt_template.format(context=context)

    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    # Generate the output
    try:
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,  # Limit the length of the generated text
            num_return_sequences=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id # Set pad_token_id to eos_token_id for open-end generation
        )
        
        # Decode the generated text
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract only the newly generated part (after the prompt)
        response = generated_text[len(prompt):].strip()
        
        # Parse the response to separate instruction and output
        if "**Generated Instruction:**" in response and "**Generated Output:**" in response:
            instruction_start = response.find("**Generated Instruction:**") + len("**Generated Instruction:**")
            output_start = response.find("**Generated Output:**")
            instruction = response[instruction_start:output_start].strip()
            output = response[output_start + len("**Generated Output:**):].strip()
            return instruction, output
        else:
            # Fallback if the model doesn't follow the format perfectly
            return "Could not parse.", response
            
    except Exception as e:
        print(f"An error occurred during generation: {e}")
        return None, None


Now, run the cell below. It will ask you how many examples you want to generate. The code will then loop that many times, each time selecting a random chunk of your source text to generate a new data point.

In [ ]:
num_examples_str = input("How many examples would you like to generate? ")
num_examples = int(num_examples_str) if num_examples_str.isdigit() else 0

alpaca_dataset = []
chunk_size = 1500 # Characters per chunk, to fit into the prompt

if num_examples > 0 and len(full_text) > chunk_size:
    print(f"\nStarting generation of {num_examples} examples...")
    for i in range(num_examples):
        # Select a random chunk of text
        start_index = torch.randint(0, len(full_text) - chunk_size, (1,)).item()
        context_chunk = full_text[start_index : start_index + chunk_size]

        # Generate instruction and output
        instruction, output = generate_instruction_output(context_chunk, model, tokenizer)

        if instruction and output and instruction != "Could not parse.":
            alpaca_dataset.append({
                "instruction": instruction,
                "input": "", # Alpaca format often has an input field, which we leave empty
                "output": output
            })
            print(f"Generated example {i + 1}/{num_examples}")
        else:
            print(f"Failed to generate or parse example {i + 1}. Skipping.")
            
    print("\nDataset generation complete!")
else:
    if num_examples <= 0:
        print("Invalid number of examples. Please enter a positive number.")
    else:
        print("Source text is not long enough to generate examples.")

## 6. Save the Dataset

Finally, we save the generated dataset into multiple formats. The `alpaca_dataset.json` file is the most important one, as it follows the standard format that can be used to fine-tune other models.

In [ ]:
def save_as_json(dataset, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, indent=4, ensure_ascii=False)
    print(f"Dataset saved to {filename}")

def save_as_txt(dataset, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for item in dataset:
            f.write(f"Instruction: {item['instruction']}\n")
            f.write(f"Output: {item['output']}\n")
            f.write("-"*40 + "\n")
    print(f"Dataset saved to {filename}")
    
def save_as_pdf(dataset, filename):
    pdf = FPDF()
    pdf.add_page()
    # Add a Unicode-supporting font
    pdf.add_font('DejaVu', '', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', uni=True)
    pdf.set_font('DejaVu', '', 12)
    
    pdf.cell(0, 10, txt="Generated Alpaca Dataset", ln=True, align='C')
    pdf.ln(5)

    for item in dataset:
        pdf.set_font('DejaVu', 'B', 12)
        pdf.multi_cell(0, 10, f"Instruction: {item['instruction']}")
        pdf.set_font('DejaVu', '', 12)
        pdf.multi_cell(0, 10, f"Output: {item['output']}")
        pdf.cell(0, 5, "-"*80, ln=True, align='C')
        pdf.ln(5)

    pdf.output(filename)
    print(f"Dataset saved to {filename}")

In [ ]:
if alpaca_dataset:
    # Save as JSON (standard Alpaca format)
    save_as_json(alpaca_dataset, "alpaca_dataset.json")
    
    # Save as TXT
    save_as_txt(alpaca_dataset, "alpaca_dataset.txt")
    
    # Save as PDF
    # Note: This requires a font file that supports Unicode characters to prevent errors.
    # We've added a common path for Linux systems. If this fails, you may need to adjust the path.
    try:
        save_as_pdf(alpaca_dataset, "alpaca_dataset.pdf")
    except Exception as e:
        print(f"Could not save PDF: {e}")
        print("This might be due to a missing font file. Please check the path in the 'save_as_pdf' function.")
else:
    print("No dataset was generated, so no files were saved.")

## 7. Conclusion

You have successfully generated an Alpaca-formatted dataset! The output files (`alpaca_dataset.json`, `alpaca_dataset.txt`, and `alpaca_dataset.pdf`) are available in the output directory of this notebook.